# OpenTof Demo: VOCUS Analysis

This notebook demonstrates the complete analysis workflow for VOCUS TOF-MS data.

## Workflow Steps (analogous to Tofware HRTS workflow)

1. Setup and Imports
2. Load data from directory
3. Mass calibration
4. Define Reference Spectrum
5. Baseline Determination
6. Peak Width Characterization
7. Peak Shape Extraction
8. Mass Calibration with custom peak shape
9. Define peak list
10. Constrained Peak Fitting
11. Save Results
12. Visualize peak fitting outputs

## 1. Setup and Imports

In [ ]:
import os
import time
import pickle
import importlib

import opentof as ot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plot_toggle = True
# %matplotlib ipympl

## 2. Load Data

Initialize OpenTof with the directory containing your H5 files. OpenTof will automatically:
- Discover all .h5 files in the directory
- Validate file compatibility
- Load and combine data (currently loading all data at once in memory)
- Determine instrument type
- Calculate first-guess mass axis

In [ ]:
# Change to the directory that contains your h5 files.
# deployment_dir_path = r"path/to/demo_data"
deployment_dir_path = r"path/to/demo_data"
d = ot.Deployment.from_directory(deployment_dir_path)

## 3. Mass Calibration

Load calibrant peaks and perform mass calibration (using Gaussian peak shape).

In [ ]:
# Load mass calibration calibrants, this will force opentof to use these instead of the defaults below
mass_calibration_filepath = "sample_MassCal_PTR.cal"
calibrants = ot.mass_calibration.load_calibrants(mass_calibration_filepath)
calibrants

In [ ]:
# Run mass calibration
calibrants = {
    'C3H5+': 41.038577,
    'NO2+': 45.992355,
    'C3H7O+': 59.049141,
    'HS8+': 256.783846,
    'C10H30O5Si5H+': 371.101233
}

d.mass_calibration(calibrants, plot_flag=plot_toggle, averaging_interval=300)
d.calibration.keys()

## 4. Define Reference Spectrum: used in the next 3 steps

Select the most stable time window for generating a reference spectrum.

Plot 1: a time-series of peak position stability of selected ion. The algorithm tries to find a stable time period.

Plot 2: the average spectrum of the chosen stable time period.

In [ ]:
d.determine_reference_spectrum(plot_flag=plot_toggle, reference_peak_mass=ot.return_mass("C3H6OH+"))
d.reference.keys()

## 5. Baseline Determination

A smoothing algorithm is applied to reference spectrum to show current baseline calculation.

Parameters have to be modified if baseline is not satisfactory.


In [ ]:
d.determine_baseline(plot_flag=plot_toggle)
d.baseline.keys()

## 6. Peak Width Characterization

Plot left: FWHM vs m/z, using RANSAC algorithm to filter out outliers, remaining points describe resolution.

Plot right: resolution vs m/z.

In [ ]:
d.determine_peak_width(plot_flag=plot_toggle)
print(d.peak_width_function(78))

## 7. Peak Shape Extraction

Extract empirical peak shape from isolated peaks in the reference spectrum.

In [ ]:
d.determine_peak_shape(plot_flag=plot_toggle, tail_intensity_cutoff=0.03)
d.custom_peak_shape

## 8. Mass Calibration with custom peak shape just defined above

In [ ]:
d.mass_calibration(
    calibrants, 
    peak_type='custom',
    custom_shape=d.custom_peak_shape,
    averaging_interval=300,
    plot_flag=plot_toggle,
)

## 9. Define peak list

This will be used for peak fitting and creation of high-resolution time series.

In [ ]:
# # Either define manually (list of strings/floats) 
# expected_compounds = [
#     "C3H6OH+", "C3H7O(H2O)+", "C6H6+", 78.062485,
#     "C7H9+", "C8H11+", "C9H13+", "C13H20H+", 
#     "C6H18Si3O3H+", "C8H25Si4O4H+", "C10H31Si5O5+"
# ]

# Or load from peak list!
df, peaks, skipped_peaks = ot.load_peak_list("sample_peaklist_PTR_large.txt",)

d.populate_peak_list_and_isotopes(peaks)
d.peak_list

## 10. Constrained Peak Fitting

In [ ]:
d.FFI_constrained(peak_type='custom') # For Fully Constrained Fitting
# d.FFI_unconstrained(peak_type='custom') # For Unconstrained Fitting With Bounds

## 11. Save Results

In [ ]:
filepath = "output/"

# Saves both "_p" and "IF" files
d.export_to_h5(output_dir=filepath, driver="PIF") # other options include "P", "IF", "OT"

## 12. Visualize peak fitting outputs

In [ ]:
# Display peak fitting outputs
d.peak_data

In [ ]:
# Plot the peak area for a selected peak
d.plot_peak_area_for("C3H6OH+")

In [ ]:
# Plot the peak height for a selected peak
d.plot_peak_amplitude_for("C6H6+")